# F1 pole position predictor

## Import


In [1]:
import pandas as pd
import sklearn
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

qualifying = pd.read_csv("F1(1950 - 2024) data\\raw\\qualifying.csv" , na_values=['\\N'])
races = pd.read_csv("F1(1950 - 2024) data\\raw\\races.csv", na_values=['\\N'])
drivers = pd.read_csv("F1(1950 - 2024) data\\raw\\drivers.csv", na_values=['\\N'])
constructors = pd.read_csv("F1(1950 - 2024) data\\raw\\constructors.csv", na_values=['\\N'])

print('Data loaded successfully')

Data loaded successfully


## Checking data

In [2]:
print(qualifying.shape)
qualifying.head()

(10494, 9)


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


## check null

In [3]:
qualifying.isnull().sum()

qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                156
q2               4647
q3               6865
dtype: int64

In [4]:
races.sample(5)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
763,764,1959,9,63,United States Grand Prix,1959-12-12,NaN,http://en.wikipedia.org/wiki/1959_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
229,230,1996,7,4,Spanish Grand Prix,1996-06-02,NaN,http://en.wikipedia.org/wiki/1996_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
772,773,1958,9,59,Portuguese Grand Prix,1958-08-24,NaN,http://en.wikipedia.org/wiki/1958_Portuguese_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
253,254,1995,15,28,Pacific Grand Prix,1995-10-22,NaN,http://en.wikipedia.org/wiki/1995_Pacific_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177,178,1999,4,6,Monaco Grand Prix,1999-05-16,NaN,http://en.wikipedia.org/wiki/1999_Monaco_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data manipulation

In [5]:
df = pd.merge(qualifying, races, how='left', on='raceId')
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
8696,8737,1052,822,131,77,3,1:31.200,1:30.186,1:29.586,2021,1,3,Bahrain Grand Prix,2021-03-28,15:00:00,http://en.wikipedia.org/wiki/2021_Bahrain_Gran...,2021-03-26,NaN,2021-03-26,NaN,2021-03-27,NaN,2021-03-27,NaN,NaN,NaN
422,423,38,25,3,17,11,1:33.759,1:32.915,NaN,2007,3,3,Bahrain Grand Prix,2007-04-15,11:30:00,http://en.wikipedia.org/wiki/2007_Bahrain_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8213,8237,1023,830,9,33,20,NaN,NaN,NaN,2019,14,14,Italian Grand Prix,2019-09-08,13:10:00,http://en.wikipedia.org/wiki/2019_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7374,7397,981,828,15,9,18,1:41.732,NaN,NaN,2017,13,14,Italian Grand Prix,2017-09-03,12:00:00,http://en.wikipedia.org/wiki/2017_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7300,7323,978,822,131,77,4,1:39.698,1:28.732,1:27.376,2017,10,9,British Grand Prix,2017-07-16,12:00:00,http://en.wikipedia.org/wiki/2017_British_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop pre 2005

In [6]:
df = df[df['year'] >= 2005]

## Convert Str to seconds

In [7]:
df['q1'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [8]:
df['q1'] = df['q1'].str.split(':').str[0].astype(float) * 60 + df['q1'].str.split(':').str[1].astype(float)

df['q2'] = df['q2'].str.split(':').str[0].astype(float) * 60 + df['q2'].str.split(':').str[1].astype(float)

df['q3'] = df['q3'].str.split(':').str[0].astype(float) * 60 + df['q3'].str.split(':').str[1].astype(float)


In [9]:
df['q1'].head(10)

0    86.572
1    86.103
2    85.664
3    85.994
4    85.960
5    86.427
6    86.295
7    86.381
8    86.919
9    86.702
Name: q1, dtype: float64

## Filling nulls with per year medians

In [10]:
df['q1'] = df.groupby('year')['q1'].transform(lambda x: x.fillna(x.median()))
df['q2'] = df.groupby('year')['q2'].transform(lambda x: x.fillna(x.median()))
df['q3'] = df.groupby('year')['q3'].transform(lambda x: x.fillna(x.median()))


In [11]:
df[['q1', 'q2', 'q3']].isnull().sum()

q1      0
q2      0
q3    376
dtype: int64

## Filling q3 nulls with 0

In [12]:
df['q3'] = df['q3'].fillna(0)

## create reached_q2&q3 column

In [13]:
df['reached_q2'] = (df['q2'].notnull()).astype(int)
df['reached_q3'] = (df['q3'].notnull()).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3
5197,5200,869,819,206,25,21,79.220,95.725,95.6570,2012,10,10,German Grand Prix,2012-07-22,12:00:00,http://en.wikipedia.org/wiki/2012_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
5818,5821,898,4,6,3,6,98.929,97.368,97.3760,2013,18,69,United States Grand Prix,2013-11-17,19:00:00,http://en.wikipedia.org/wiki/2013_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
577,578,45,25,3,17,12,92.173,91.996,83.4460,2007,10,20,European Grand Prix,2007-07-22,12:00:00,http://en.wikipedia.org/wiki/2007_European_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
3176,3178,10,22,23,23,13,81.558,81.222,92.3950,2009,10,11,Hungarian Grand Prix,2009-07-26,12:00:00,http://en.wikipedia.org/wiki/2009_Hungarian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
9929,9987,1116,817,213,3,15,96.213,95.974,84.9105,2023,18,69,United States Grand Prix,2023-10-22,19:00:00,https://en.wikipedia.org/wiki/2023_United_Stat...,2023-10-20,17:30:00,2023-10-21,18:00:00,NaN,NaN,2023-10-20,21:00:00,2023-10-21,22:00:00,1,1


In [14]:
print(df[df['q3'].isnull()]['reached_q3'].value_counts())

Series([], Name: count, dtype: int64)


## create got_pole column

In [15]:
df['got_pole'] = (df['position'] == 1).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
9562,9620,1096,839,214,31,8,85.735,85.007,84.830,2022,22,24,Abu Dhabi Grand Prix,2022-11-20,13:00:00,http://en.wikipedia.org/wiki/2022_Abu_Dhabi_Gr...,2022-11-18,10:00:00,2022-11-18,13:00:00,2022-11-19,11:00:00,2022-11-19,14:00:00,NaN,NaN,1,1,0
6129,6132,911,825,1,20,7,131.081,128.901,128.679,2014,12,13,Belgian Grand Prix,2014-08-24,12:00:00,http://en.wikipedia.org/wiki/2014_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
8175,8199,1022,20,6,5,2,104.109,103.037,103.267,2019,13,13,Belgian Grand Prix,2019-09-01,13:10:00,http://en.wikipedia.org/wiki/2019_Belgian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
10400,10458,1140,825,210,20,7,77.125,77.003,76.886,2024,20,32,Mexico City Grand Prix,2024-10-27,20:00:00,https://en.wikipedia.org/wiki/2024_Mexico_City...,2024-10-25,18:30:00,2024-10-25,22:00:00,2024-10-26,17:30:00,2024-10-26,21:00:00,NaN,NaN,1,1,0
1238,1239,82,8,1,9,1,74.320,89.159,0.000,2005,12,10,German Grand Prix,2005-07-24,14:00:00,http://en.wikipedia.org/wiki/2005_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1


In [16]:
df['got_pole'].value_counts()

got_pole
0    7872
1     394
Name: count, dtype: int64

In [17]:
df.groupby('year')['q2'].median().head(20)

year
2005    89.1590
2006    81.4840
2007    83.1205
2008    83.4555
2009    91.2220
         ...   
2020    79.9620
2021    81.0860
2022    85.2250
2023    84.4405
2024    83.8770
Name: q2, Length: 20, dtype: float64

In [18]:
for i, col in enumerate(df.columns):
    print(i, col)

pd.set_option('display.max_columns', None)
df.sample(20)

0 qualifyId
1 raceId
2 driverId
3 constructorId
4 number
5 position
6 q1
7 q2
8 q3
9 year
10 round
11 circuitId
12 name
13 date
14 time
15 url
16 fp1_date
17 fp1_time
18 fp2_date
19 fp2_time
20 fp3_date
21 fp3_time
22 quali_date
23 quali_time
24 sprint_date
25 sprint_time
26 reached_q2
27 reached_q3
28 got_pole


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
6995,7018,963,13,3,19,10,95.267,94.422,94.6710,2016,16,2,Malaysian Grand Prix,2016-10-02,07:00:00,http://en.wikipedia.org/wiki/2016_Malaysian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
3674,3676,351,2,15,22,15,108.696,108.557,90.7050,2010,15,15,Singapore Grand Prix,2010-09-26,12:00:00,http://en.wikipedia.org/wiki/2010_Singapore_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
9109,9150,1072,832,6,55,15,88.237,113.652,81.0545,2021,21,77,Saudi Arabian Grand Prix,2021-12-05,17:30:00,http://en.wikipedia.org/wiki/2021_Saudi_Arabia...,2021-12-03,NaN,2021-12-03,NaN,2021-12-04,NaN,2021-12-04,NaN,NaN,NaN,1,1,0
9416,9474,1088,844,6,16,2,71.443,70.988,70.3630,2022,15,39,Dutch Grand Prix,2022-09-04,13:00:00,http://en.wikipedia.org/wiki/2022_Dutch_Grand_...,2022-09-02,10:30:00,2022-09-02,14:00:00,2022-09-03,10:00:00,2022-09-03,13:00:00,NaN,NaN,1,1,0
8039,8063,1015,825,210,20,6,71.865,71.363,71.1090,2019,6,6,Monaco Grand Prix,2019-05-26,13:10:00,http://en.wikipedia.org/wiki/2019_Monaco_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5768,5771,895,824,206,22,22,94.958,91.989,90.9080,2013,15,22,Japanese Grand Prix,2013-10-13,06:00:00,http://en.wikipedia.org/wiki/2013_Japanese_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
370,371,36,2,2,9,3,86.895,85.358,86.5560,2007,1,1,Australian Grand Prix,2007-03-18,03:00:00,http://en.wikipedia.org/wiki/2007_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
7940,7964,1010,844,6,16,5,82.017,81.739,81.4420,2019,1,1,Australian Grand Prix,2019-03-17,05:10:00,http://en.wikipedia.org/wiki/2019_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
9909,9967,1115,807,210,27,15,85.904,85.783,84.9105,2023,17,78,Qatar Grand Prix,2023-10-08,17:00:00,https://en.wikipedia.org/wiki/2023_Qatar_Grand...,2023-10-06,13:30:00,2023-10-07,13:00:00,NaN,NaN,2023-10-06,17:00:00,2023-10-07,17:30:00,1,1,0


## Features and Target

In [19]:
features = ['q1', 'q2', 'q3', 'reached_q2', 'reached_q3', 'year', 'circuitId', 'constructorId', 'driverId', 'round']
target = 'got_pole'

## New df

In [20]:
df_cleaned = df[features + [target]]
print(df_cleaned.shape)
df_cleaned.sample(5)

(8266, 11)


,q1,q2,q3,reached_q2,reached_q3,year,circuitId,constructorId,driverId,round,got_pole
4910,88.565,91.532,90.3075,1,1,2011,68,205,5,17,0
9350,66.613,85.225,88.0995,1,1,2022,70,1,817,11,0
118,76.285,75.907,86.7915,1,1,2008,6,7,10,6,0
8652,90.138,79.962,81.0110,1,1,2020,3,210,154,15,0
5979,88.155,87.685,94.4715,1,1,2014,4,10,807,5,0


In [21]:
print(df_cleaned.shape)
print(df_cleaned['got_pole'].value_counts())
print(df_cleaned.isnull().sum())

(8266, 11)
got_pole
0    7872
1     394
Name: count, dtype: int64
q1               0
q2               0
q3               0
reached_q2       0
reached_q3       0
                ..
circuitId        0
constructorId    0
driverId         0
round            0
got_pole         0
Length: 11, dtype: int64


## sklearn

In [22]:
from sklearn.model_selection import train_test_split

x = df_cleaned[features]
y = df_cleaned[target]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape, x_test.shape)


(6612, 10) (1654, 10)


In [29]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced',max_iter=1000)
model.fit(x_train, y_train)
print(model.classes_)

[0 1]


c:\Users\Tahsin\anaconda3\envs\data_analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [30]:
from sklearn.metrics import classification_report

y_pred = model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.59      0.74      1575
           1       0.08      0.75      0.15        79

    accuracy                           0.60      1654
   macro avg       0.53      0.67      0.44      1654
weighted avg       0.94      0.60      0.71      1654

